# Class 7: Build an AI Interview Coach with LangChain Prompt Templates
**Soumya AI Vision | Notebook + Streamlit project**


## 01. Problem and architecture

A fixed prompt cannot conveniently adapt to job roles, years of experience or difficulty. We will parameterize prompts and then connect them to a UI. The prompt is an instruction template—not a guarantee of correctness.

**Try it:** Run this cell and inspect the output before continuing.

## 02. Install the dependencies

Run the install command in Colab. In a terminal, omit the `%` magic and use `pip install`. For the final app, Streamlit and langchain-openai are also required.

**Try it:** Run this cell and inspect the output before continuing.

In [1]:
%pip install -q langchain-core langchain-openai langchain-groq streamlit

/Users/soumyaranjannayak/Developer/personal/SoumyaAI_Vision/projects/Interview_Coach/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 03. Import prompt classes

`PromptTemplate` handles text templates, while `ChatPromptTemplate` structures messages. `MessagesPlaceholder` inserts an existing list of conversation messages.

**Try it:** Run this cell and inspect the output before continuing.

In [2]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

## 04. string concatenation

Manual concatenation works for tiny examples but becomes awkward for repeated input fields.


In [3]:
role = "GenAI Engineer"
difficulty = "Intermediate"
manual_prompt = "Ask one " + difficulty + " question for a " + role
print(manual_prompt)

Ask one Intermediate question for a GenAI Engineer


## 05. PromptTemplate: reusable variables

Curly braces mark required variables. `.format()` returns a formatted string, not a model-generated answer.

**Try it:** Run this cell and inspect the output before continuing.

In [4]:
interview_prompt = PromptTemplate.from_template("Ask one {difficulty} question for a {role}.")
print(interview_prompt.format(difficulty="Beginner", role="Python Developer"))
print(interview_prompt.format(difficulty="Advanced", role="GenAI Engineer"))

Ask one Beginner question for a Python Developer.
Ask one Advanced question for a GenAI Engineer.


## 06. Inspect required inputs

`input_variables` helps you debug missing parameters before sending a request.

**Try it:** Run this cell and inspect the output before continuing.

In [5]:
print(interview_prompt.input_variables)
# Demonstrate missing keys carefully:
try:
    interview_prompt.format(role="Data Scientist")
except KeyError as error:
    print("Missing input:", error)

['difficulty', 'role']
Missing input: 'difficulty'


## 07. Partial variables

`.partial()` locks in values you reuse frequently; the remaining variables can still change.

**Try it:** Run this cell and inspect the output before continuing.

In [6]:
fixed_prompt = interview_prompt.partial(role="GenAI Engineer")
print(fixed_prompt.input_variables)
print(fixed_prompt.format(difficulty="Intermediate"))
print(fixed_prompt.format(difficulty="Advanced"))

['difficulty']
Ask one Intermediate question for a GenAI Engineer.
Ask one Advanced question for a GenAI Engineer.


## 08. Why chat roles matter

A system message describes assistant behavior and a human message describes the current task. Message roles are more expressive than concatenating one long string.



In [7]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional {role} interviewer."),
    ("human", "Ask one {difficulty} question about {topic}."),
])
messages = chat_prompt.format_messages(role="Python Developer", difficulty="Beginner", topic="lists")
for message in messages:
    print(type(message).__name__, "=>", message.content)

SystemMessage => You are a professional Python Developer interviewer.
HumanMessage => Ask one Beginner question about lists.


## 09. MessagesPlaceholder for history

Insert previous `HumanMessage` and `AIMessage` objects without manually rebuilding role blocks. This is context supplied by your program; it is not persistent storage.



In [8]:
conversation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a supportive technical interviewer."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])
history = [HumanMessage(content="I have 2 years of Python experience."),
           AIMessage(content="Great. Let's discuss Python functions.")]
formatted = conversation_prompt.format_messages(history=history, question="Ask my next question.")
for message in formatted:
    print(type(message).__name__, "=>", message.content)

SystemMessage => You are a supportive technical interviewer.
HumanMessage => I have 2 years of Python experience.
AIMessage => Great. Let's discuss Python functions.
HumanMessage => Ask my next question.


## 10. Preview the final interview template

This is the central prompt used by the Streamlit application. Notice that role, experience, difficulty and interview type are separate inputs.



In [9]:
system_template = """You are a professional technical interviewer.
Role: {role}. Experience: {experience}. Difficulty: {difficulty}.
Interview type: {interview_type}. Ask one question at a time.
After each candidate answer, provide concise actionable feedback."""
coach_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{request}"),
])
preview = coach_prompt.format_messages(role="GenAI Engineer", experience="2–4 years",
    difficulty="Intermediate", interview_type="Technical + Coding",
    history=[], request="Begin with one question.")
for msg in preview:
    print(type(msg).__name__, "=>", msg.content)

SystemMessage => You are a professional technical interviewer.
Role: GenAI Engineer. Experience: 2–4 years. Difficulty: Intermediate.
Interview type: Technical + Coding. Ask one question at a time.
After each candidate answer, provide concise actionable feedback.
HumanMessage => Begin with one question.


## 11. Optional live model call

Requires `OPENAI_API_KEY` set outside the notebook. `prompt | model` is a simple pipeline preview; we will teach LCEL in detail in Class 8. Do not display real credentials while recording.



In [10]:
import os
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    chain = coach_prompt | llm
    reply = chain.invoke({"role": "GenAI Engineer", "experience": "2–4 years",
        "difficulty": "Intermediate", "interview_type": "Technical + Coding",
        "history": [], "request": "Begin with one interview question only."})
    print(reply.content)
else:
    print("Skipping optional live demo: configure OPENAI_API_KEY securely.")

Sure! Here’s your first question:

**Question 1:** Can you explain the difference between supervised and unsupervised learning in the context of machine learning? Please provide examples of each.


## 11A. Choose the LLM provider — OpenAI **or Groq**
The **same `ChatPromptTemplate` works with both providers** because prompt formatting is independent of model inference. Groq offers free-tier access subject to limits. Create a key in [Groq Console](https://console.groq.com/keys) and store it securely in `GROQ_API_KEY`; OpenAI uses `OPENAI_API_KEY`. Never publish keys.


In [13]:
import os
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

provider = "groq"  # Change to "openai" for comparison.
api_key_var = "GROQ_API_KEY" if provider == "groq" else "OPENAI_API_KEY"
model_id = "qwen/qwen3.8-27b" if provider == "groq" else "gpt-4o-mini"

if os.getenv(api_key_var):
    llm = (ChatGroq(model=model_id, temperature=0.3)
           if provider == "groq" else ChatOpenAI(model=model_id, temperature=0.3))
    sample = chat_prompt.format_messages(role="Python Developer", difficulty="Beginner",topic="lists")
    response = llm.invoke(sample)
    print(response.content)
else:
    print(f"Set {api_key_var} safely in your environment to run {provider}.")


Here is a beginner-level question about Python lists:

**Question:**
What is the output of the following code snippet?

```python
my_list = [1, 2, 3, 4, 5]
my_list.append(6)
print(my_list)
```

**Please provide your answer.**


## 11B. Groq configuration and real-world notes
1. Generate an API key in the Groq console and use Colab Secrets or environment variables, never hardcoded source.
2. Install `langchain-groq` and import `ChatGroq`.
3. Instantiate with `ChatGroq(model="lqwen/qwen3.8-27b")`.
4. A free tier is subject to usage limits, and model availability can change.
5. **Groq isn't a new prompt-template type:** the prompt is the same; the model implementation is swapped.

Switch `provider` and compare results. Consider correctness, latency and output style.


## 12. Streamlit architecture: inputs and session state

The app sidebar captures role, experience, difficulty, interview type and question count. `st.session_state` retains chat messages during Streamlit reruns. It is session-level memory, not a database.



## 13. Streamlit architecture: model and prompt pipeline

The `ask_model()` function assembles template inputs, passes current conversation history, and calls the chat model. The AI produces the question or feedback; Streamlit only displays it.



In [ ]:
# The complete implementation is in streamlit_app_groq_openai.py.
# Read the functions `ask_model()` and `build_model()` in that file.
print("Prompt + model = a reusable interview request pipeline")

## 14. Launch and demonstrate the app

Save `streamlit_app_groq_openai.py` beside the notebook (or download it into a local project folder). Configure the API key as an environment variable or Streamlit secret. Run `streamlit run streamlit_app_groq_openai.py` in a local terminal. Demonstrate a full interview with two question settings.



In [ ]:
print("Terminal commands:")
print("pip install streamlit langchain-core langchain-openai langchain-groq")
print("streamlit run streamlit_app_groq_openai.py")

## 15. Test scenarios and limitations

Check changing role, restarting, last-question feedback, missing API key, and model failure. The model may still hallucinate or ignore instructions; a prompt template is not a validation or grading system. Production apps need stronger evaluation, safety and storage.



In [ ]:
test_cases = [
    "Start with Python Developer / Fresher / Beginner",
    "Restart with GenAI Engineer / 5+ years / Advanced",
    "Answer the last question and confirm no extra question appears",
    "Unset API key and confirm the warning",
    "Change sidebar settings mid-interview and confirm the warning",
]
for case in test_cases:
    print("CHECK:", case)